# 03 — Verdict final (Amendement-01)

**Logique de décision révisée** — remplace l'ancienne logique ΔAIC
simple par les branches ordonnées de l'amendement §5-§6.

## Ce qui a changé
- CPL n'entre plus dans le verdict (calculé pour contexte seulement)
- Pivot = σ_φ (depuis `02_freeform_wz_reconstruction.ipynb`)
- ΔDIC remplace ΔAIC comme critère secondaire
- Flag P (pathologie φ̇² > 0.95) comme garde-fou
- Verdict par cellule (potentiel × SNe) puis verdict global (§6)

## Entrées nécessaires
- `results/sigma_phi.json` (notebook 02)
- 4 chaînes tachyon (run01-04) + 2 chaînes ΛCDM de référence (run00, run00b)

In [5]:
import sys, os, json
REPO = os.path.expanduser("~/Desktop/souriau-falsification")
os.chdir(REPO)
sys.path.insert(0, REPO)

import numpy as np
from getdist import loadMCSamples

RUNS = {
    ('exponentiel', 'Pantheon+'): {
        'tachyon': 'cobaya_runs/tachyon_output/run05_expo_pantheon_full/full_expo_pantheon',
        'lcdm':    'cobaya_runs/tachyon_output/run00c_lcdm_reference_pantheon/full_lcdm_pantheon',
    },
    #('exponentiel', 'DES-SN5YR'): {
        #'tachyon': 'cobaya_runs/tachyon_output/run06_expo_des_full/full_expo_des',
        #'lcdm':    'cobaya_runs/tachyon_output/run00b_lcdm_reference_desl/full_lcdm_reference_des',
    #},
    #('inverse', 'Pantheon+'): {
        #'tachyon': 'cobaya_runs/tachyon_output/run03_invpower_pantheon_full/full_invpower_pantheon',
        #'lcdm':    'cobaya_runs/tachyon_output/run00c_lcdm_reference_pantheon/full_lcdm_pantheon',
    #},
    ('inverse', 'DES-SN5YR'): {
        'tachyon': 'cobaya_runs/tachyon_output/run04_invpower_des_full/full_invpower_des',
        'lcdm':    'cobaya_runs/tachyon_output/run00b_lcdm_reference_des/full_lcdm_reference_des',
    }
}

SEUIL_SIGMA_HIGH = 3.0   # franchissement physique robuste
SEUIL_SIGMA_LOW  = 2.0   # pas de franchissement robuste
SEUIL_DIC_LOW    = 2.0   # équivalence/préférence tachyon
SEUIL_DIC_HIGH   = 5.0   # rejet forme du modèle
SEUIL_PHIDOT2    = 0.95  # flag P: champ devient poussière

print("Setup OK — 4 cellules définies")

Setup OK — 4 cellules définies


In [6]:
import json, os

sigma_phi_path = 'results/sigma_phi.json'

if os.path.exists(sigma_phi_path):
    with open(sigma_phi_path) as f:
        sigma_phi_data = json.load(f)
    print("σ_φ chargé depuis results/sigma_phi.json :")
    for sne, data in sigma_phi_data.items():
        print(f"  {sne:<12} : σ_φ = {data['sigma_phi']}")
else:
    print("results/sigma_phi.json n'existe pas encore.")
    print("    Le notebook 02_freeform_wz_reconstruction.ipynb")
    print("    n'a pas encore été exécuté avec des chaînes complètes.")
    print("    On continue avec sigma_phi = None (verdict partiel).")
    sigma_phi_data = {
        'Pantheon+': {'sigma_phi': None},
        'DES-SN5YR': {'sigma_phi': None},
    }

σ_φ chargé depuis results/sigma_phi.json :
  Pantheon+    : σ_φ = 0.013913051532569057


In [21]:
import sys, os
import numpy as np
import pandas as pd
from getdist import loadMCSamples

sys.path.insert(0, REPO)
from src.tachyon.potentials import ExponentialPotential, InversePowerPotential
from src.tachyon.rolling_tachyon import build_tachyon_background

def load_chain_pandas(path, burn_in_frac=0.3):
    """
    Charge une chaîne cobaya directement en pandas, contourne
    le bug getdist sur deleteFixedParams() pour les chaînes
    avec des paramètres entièrement fixes (tachyon_lambda etc.).
    """
    with open(path + '.1.txt') as f:
        header = f.readline().strip().lstrip('#').split()

    df = pd.read_csv(path + '.1.txt', sep=r'\s+',
                      comment='#', names=header)

    n_burn = int(len(df) * burn_in_frac)
    return df.iloc[n_burn:].reset_index(drop=True)


def compute_dic_from_df(df):
    """DIC = chi2_mean + 2*(chi2_mean - chi2_min), depuis un DataFrame."""
    chi2_chain = df['chi2'].values
    chi2_mean = float(np.mean(chi2_chain))
    chi2_min  = float(np.min(chi2_chain))
    dic = chi2_mean + 2 * (chi2_mean - chi2_min)
    return dic, chi2_mean, chi2_min


def compute_dic_from_getdist(samp):
    """DIC depuis un objet getdist MCSamples (cas normal, params variables)."""
    names = [p.name for p in samp.getParamNames().names]
    idx_chi2 = names.index('chi2')
    chi2_chain = samp.samples[:, idx_chi2]
    chi2_mean = float(np.mean(chi2_chain))
    chi2_min  = float(np.min(chi2_chain))
    dic = chi2_mean + 2 * (chi2_mean - chi2_min)
    return dic, chi2_mean, chi2_min


def compute_phidot_max_bestfit(samp, potentiel):
    """
    Recalcule phi_dot_max a posteriori depuis les valeurs best-fit
    (utile si phidot_max n'a pas été sauvegardé dans le YAML d'origine).
    """
    if potentiel == 'exponentiel':
        lam = samp.mean('tachyon_lambda')
        pot = ExponentialPotential(V0=1.0, lam=lam)
    else:
        n = samp.mean('tachyon_n')
        pot = InversePowerPotential(V0=1.0, n=n)

    phi_init = samp.mean('tachyon_phi_init')
    Om = samp.mean('Omega_m')

    bg = build_tachyon_background(pot, Om, phi_init=phi_init,
                                    x_init=0.0, z_init=50.0)
    return float(np.max(np.abs(bg['phi_dot'])))

resultats = {}

for (potentiel, sne), paths in RUNS.items():
    if not os.path.exists(os.path.dirname(paths['tachyon'])):
        print(f"⚠️  Run manquant : {potentiel} × {sne}")
        continue

    samp_t = loadMCSamples(paths['tachyon'], settings={'ignore_rows': 0.3})
    names_t = [p.name for p in samp_t.getParamNames().names]
    dic_t, mean_t, min_t = compute_dic_from_getdist(samp_t)

    df_lcdm = load_chain_pandas(paths['lcdm'])
    dic_l, mean_l, min_l = compute_dic_from_df(df_lcdm)

    delta_dic = dic_t - dic_l

    if 'phidot_max' in names_t:
        phidot_max = samp_t.mean('phidot_max')
        source_phidot = "sauvegardé"
    else:
        phidot_max = compute_phidot_max_bestfit(samp_t, potentiel)
        source_phidot = "recalculé (best-fit)"

    flag_P = bool(phidot_max**2 > SEUIL_PHIDOT2)

    resultats[(potentiel, sne)] = {
        'dic_tachyon': dic_t, 'dic_lcdm': dic_l, 'delta_dic': delta_dic,
        'phidot_max': phidot_max, 'flag_P': flag_P,
        'sigma_phi': sigma_phi_data.get(sne, {}).get('sigma_phi'),
    }

    print(f"\n{potentiel} × {sne}")
    print(f"  DIC tachyon = {dic_t:.2f}   DIC ΛCDM = {dic_l:.2f}")
    print(f"  ΔDIC        = {delta_dic:+.2f}")
    print(f"  φ̇²_max      = {phidot_max**2:.4f}  ({source_phidot})")
    print(f"  flag P      = {flag_P}")

print()
print(f"  {len(resultats)} cellule(s) calculée(s)")


exponentiel × Pantheon+
  DIC tachyon = 2444.61   DIC ΛCDM = 2444.26
  ΔDIC        = +0.35
  φ̇²_max      = 0.1619  (recalculé (best-fit))
  flag P      = False

inverse × DES-SN5YR
  DIC tachyon = 2674.97   DIC ΛCDM = 2674.85
  ΔDIC        = +0.13
  φ̇²_max      = 0.2748  (recalculé (best-fit))
  flag P      = False

  2 cellule(s) calculée(s)


In [ ]:
def verdict_cellule(sigma_phi, delta_dic, flag_P):
    """
    Branches ORDONNÉES (amendement-01 §5).
    La première condition qui s'applique fixe le verdict.
    """
    if sigma_phi is not None and sigma_phi >= SEUIL_SIGMA_HIGH:
        return "DISFAVORISÉE", "franchissement physique"

    if flag_P:
        return "DISFAVORISÉE", "pathologie interne (champ → poussière)"

    if sigma_phi is not None and sigma_phi < SEUIL_SIGMA_LOW:
        if delta_dic <= SEUIL_DIC_LOW:
            return "SOUTENUE", "ajustement ≥ ΛCDM, pas de franchissement"
        elif delta_dic >= SEUIL_DIC_HIGH:
            return "DISFAVORISÉE", "forme du modèle (pas la borne -1)"

    return "NON CONCLUANT", "zone intermédiaire"


print("  VERDICT PAR CELLULE (potentiel × SNe)")

verdicts_cellules = {}
for (potentiel, sne), res in resultats.items():
    verdict, motif = verdict_cellule(
        res['sigma_phi'], res['delta_dic'], res['flag_P']
    )
    verdicts_cellules[(potentiel, sne)] = verdict

    sp_str = f"{res['sigma_phi']:.3f}" if res['sigma_phi'] is not None else "N/A"

    print(f"\n  {potentiel} × {sne}")
    print(f"    σ_φ = {sp_str}, ΔDIC = {res['delta_dic']:+.2f}, "
          f"flag_P = {res['flag_P']}")
    print(f"    → {verdict} ({motif})")

  VERDICT PAR CELLULE (potentiel × SNe)

  exponentiel × Pantheon+
    σ_φ = 0.014, ΔDIC = +0.35, flag_P = False
    → SOUTENUE (ajustement ≥ ΛCDM, pas de franchissement)

  inverse × DES-SN5YR
    σ_φ = N/A, ΔDIC = +0.13, flag_P = False
    → NON CONCLUANT (zone intermédiaire)


In [20]:
def verdict_global(verdicts_cellules, sigma_phi_data):
    sigma_phis = {sne: d['sigma_phi'] for sne, d in sigma_phi_data.items()}

    if all(sp is not None and sp >= 3 for sp in sigma_phis.values()):
        return "DISFAVORISÉE (franchissement physique)", \
               "σ_φ ≥ 3 pour les deux catalogues SNe"

    n_high = sum(1 for sp in sigma_phis.values() if sp is not None and sp >= 3)
    if n_high == 1:
        return "NON CONCLUANT", \
               "discordance inter-SNe sur σ_φ ≥ 3 (systématique candidate)"

    if any(v == "DISFAVORISÉE" and "pathologie" in str(verdicts_cellules)
           for v in verdicts_cellules.values()):
        return "DISFAVORISÉE (pathologie interne)", \
               "au moins une cellule porte le flag P"

    if all(v == "SOUTENUE" for v in verdicts_cellules.values()):
        return "SOUTENUE", "unanimité des 4 cellules"

    if all(v == "DISFAVORISÉE" for v in verdicts_cellules.values()):
        return "DISFAVORISÉE (forme du modèle)", "unanimité des 4 cellules"

    return "NON CONCLUANT", "zone intermédiaire ou discordance inter-cellule"


verdict_final, motif_final = verdict_global(verdicts_cellules, sigma_phi_data)

print("  VERDICT GLOBAL")
print(f"\n  {verdict_final}")
print(f"  Motif : {motif_final}")
print()
print("  Rappel : 'soutenue' et 'disfavorisée (forme)' exigent")
print("  l'unanimité des 4 cellules. Toute discordance → non concluant.")
print("  C'est le comportement voulu (amendement §6, note d'asymétrie).")

  VERDICT GLOBAL

  NON CONCLUANT
  Motif : zone intermédiaire ou discordance inter-cellule

  Rappel : 'soutenue' et 'disfavorisée (forme)' exigent
  l'unanimité des 4 cellules. Toute discordance → non concluant.
  C'est le comportement voulu (amendement §6, note d'asymétrie).


In [ ]:
import pandas as pd

rows = []
for (potentiel, sne), res in resultats.items():
    rows.append({
        'Potentiel': potentiel,
        'SNe': sne,
        'σ_φ': res['sigma_phi'],
        'ΔDIC': round(res['delta_dic'], 2),
        'flag_P': res['flag_P'],
        'Verdict cellule': verdicts_cellules[(potentiel, sne)],
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

print(f"  VERDICT GLOBAL : {verdict_final}")

# Sauvegarder
df.to_csv('results/verdict_cellules.csv', index=False)
with open('results/verdict_global.json', 'w') as f:
    json.dump({'verdict': verdict_final, 'motif': motif_final}, f, indent=2)
print("\nSauvegardé → results/verdict_cellules.csv, results/verdict_global.json")